In [ ]:
import cosmographi as cp
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from contextlib import closing
import pandas as pd
from time import time
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pick whichever GPU has more free memory
np.random.seed(0)
key = jax.random.PRNGKey(0)
colours = ["purple", "green", "red", "teal", "brown", "yellow"]

The first step is to extract the data from the LSST database using SQL.
Note that form the output, we can tell that each rown observation and each collumn is a variable. 

In [ ]:
# Extract survey data from the LSST database
# #####################################################################
survey_path = "/home/renee/renee/opsim/baseline_v5.0.1_10yrs.db"
with closing(sqlite3.connect(survey_path)) as conn:
    query = f"""
        SELECT 
            observationId,
            
            observationStartMJD,
            visitExposureTime,
            numExposures,
            airmass,
            fieldRA, 
            fieldDec, 
            rotTelPos, 
            rotSkyPos, 
            skyBrightness,
            band,
            filter,
            seeingFwhmEff,
            observation_reason
        FROM observations
        LIMIT 2000;
        """
    survey_df = pd.read_sql_query(query, conn) # all data will be togetehr in a pandas df
print(survey_df)
assert (
    survey_df["numExposures"].nunique() == 1
), "All exposures should have the same number of exposures."
survey_t = survey_df["observationStartMJD"].values
print(type(survey_t))
survey_ra = survey_df["fieldRA"].values
survey_dec = survey_df["fieldDec"].values
survey_exp_time = survey_df["visitExposureTime"].values
survey_airmass = survey_df["airmass"].values
survey_sky_brightness = survey_df["skyBrightness"].values
survey_seeing = survey_df["seeingFwhmEff"].values
survey_PSF_Aeff = 4 * np.pi * (survey_seeing / 2.355) ** 2  # convert FWHM to effective area
survey_band = survey_df["band"].values
survey_obsreason = survey_df["observation_reason"].values

      observationId  observationStartMJD  visitExposureTime  numExposures  \
0                 0         60981.002252               15.0             1   
1                 1         60981.002498               15.0             1   
2                 2         60981.002744               15.0             1   
3                 3         60981.002988               15.0             1   
4                 4         60981.003234               15.0             1   
...             ...                  ...                ...           ...   
1995           1995         60991.338102               30.0             1   
1996           1996         60991.340081               30.0             1   
1997           1997         60991.340527               30.0             1   
1998           1998         60991.340945               30.0             1   
1999           1999         60991.341370               30.0             1   

       airmass     fieldRA   fieldDec  rotTelPos   rotSkyPos  skyBrightness

In [ ]:
## Setting up a 'source' to observe
We now need to have a source to observe. In this example we have coded up a Type Ia supernova that is specified by the [SALT2 2021](https://academic.oup.com/mnras/article/504/3/4111/6225808) light curve model, and have assumed Milky Way extinction according to the [Calzetti law](https://iopscience.iop.org/article/10.1086/308692).  We are assuming a standard LCDM cosmology.


In [ ]:
# Create source

C = cp.Cosmology()
source_obj = cp.source_factory(cp.SALT2_2021, cp.source.effects.MWExtinction_Calzetti00) #creates new class 
S = source_obj(cosmology=C, A_V_c00mw=0.1, R_V_c00mw=3.1) # dust extinction parameters
S.load_salt2_model()
S.M.to_static()
S.CL.to_static()
print(S)